# 🛡️ PhishGuard AI — Model Training
Upload the PhiUSIIL Phishing URL Dataset and run all cells. This notebook trains Decision Tree, Random Forest, Naive Bayes and KNN, compares them, and saves the best Random Forest model.


In [ ]:
!pip -q install scikit-learn pandas numpy joblib
from google.colab import files
uploaded = files.upload()


In [ ]:
import io, os, json, zipfile, joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

filename=list(uploaded.keys())[0]
df=pd.read_csv(io.BytesIO(uploaded[filename]))
print('Dataset shape:', df.shape)
print(df.head(3))


In [ ]:
drop_cols=['FILENAME','URL','Domain','TLD','Title','label','URLSimilarityIndex']
drop_cols=[c for c in drop_cols if c in df.columns]
X=df.drop(columns=drop_cols).select_dtypes(include='number')
y=df['label']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print('Train:',X_train.shape,'Test:',X_test.shape)


In [ ]:
models={
'Decision Tree':DecisionTreeClassifier(max_depth=15,random_state=42),
'Random Forest':RandomForestClassifier(n_estimators=100,max_depth=15,random_state=42,n_jobs=-1),
'Naive Bayes':GaussianNB()
}
results={}
for name,m in models.items():
    m.fit(X_train,y_train)
    p=m.predict(X_test)
    results[name]={'testing_accuracy':float(accuracy_score(y_test,p)),'confusion_matrix':confusion_matrix(y_test,p).tolist()}
    print('\n',name, 'Accuracy:', results[name]['testing_accuracy'])
    print(confusion_matrix(y_test,p))


In [ ]:
scaler=StandardScaler()
Xtr=scaler.fit_transform(X_train); Xte=scaler.transform(X_test)
knn=KNeighborsClassifier(n_neighbors=5,n_jobs=-1)
knn.fit(Xtr,y_train); p=knn.predict(Xte)
results['KNN']={'testing_accuracy':float(accuracy_score(y_test,p)),'confusion_matrix':confusion_matrix(y_test,p).tolist()}
print('KNN Accuracy:',results['KNN']['testing_accuracy'])
print(confusion_matrix(y_test,p))


In [ ]:
print('\nMODEL COMPARISON')
for k,v in results.items(): print(f"{k}: {v['testing_accuracy']*100:.4f}%")

os.makedirs('phishguard_models',exist_ok=True)
joblib.dump(models['Random Forest'],'phishguard_models/phishguard_random_forest.joblib')
with open('phishguard_models/feature_columns.json','w') as f: json.dump(list(X.columns),f,indent=2)
with open('phishguard_models/model_metrics.json','w') as f: json.dump(results,f,indent=2)
print('Saved model files.')


In [ ]:
from google.colab import files
with zipfile.ZipFile('phishguard_models.zip','w',zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir('phishguard_models'):
        z.write('phishguard_models/'+fn,fn)
files.download('phishguard_models.zip')
